In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
merged_df = pd.read_csv("merged_data.csv")

In [3]:
X = merged_df[
    ["duration_sec", "category", "device", "country"]
]

y = merged_df["watch_completed"]

KeyError: "['country'] not in index"

In [4]:
merged_df["event_type"].value_counts()

event_type
View       499207
Like       124865
Comment     39915
Share       14853
Follow       9892
Name: count, dtype: int64

In [5]:
print(merged_df.columns)

Index(['event_id', 'user_id', 'video_id', 'event_type', 'timestamp',
       'watch_time_sec', 'watch_completed', 'creator_id', 'category',
       'language', 'upload_date', 'duration_sec', 'has_hashtags',
       'is_verified_creator', 'device', 'date', 'year_month'],
      dtype='object')


In [11]:
users = pd.read_csv("data/users.csv")
videos = pd.read_csv("data/videos.csv")
events = pd.read_csv("data/events.csv")

In [12]:
users.columns

Index(['user_id', 'signup_date', 'country', 'device', 'age_group', 'gender',
       'acquisition_source', 'premium_user'],
      dtype='object')

In [13]:
merged_df = events.merge(users, on="user_id", how="left")
merged_df = merged_df.merge(videos, on="video_id", how="left")

In [14]:
merged_df.columns

Index(['event_id', 'user_id', 'video_id', 'event_type', 'timestamp',
       'watch_time_sec', 'watch_completed', 'signup_date', 'country', 'device',
       'age_group', 'gender', 'acquisition_source', 'premium_user',
       'creator_id', 'category', 'language', 'upload_date', 'duration_sec',
       'has_hashtags', 'is_verified_creator'],
      dtype='object')

In [15]:
ml_df = merged_df[
    merged_df["event_type"] == "View"
]

In [16]:
X = ml_df[
    [
        "duration_sec",
        "category",
        "country",
        "language",
        "device",
        "has_hashtags",
        "is_verified_creator"
    ]
]

In [17]:
y = ml_df["watch_completed"]

In [18]:
print(X.shape)
print(y.shape)

print(X.head())
print(y.head())

(499207, 7)
(499207,)
   duration_sec   category country language   device  has_hashtags  \
0            80      Music      UK  English  Android             1   
2           156     Comedy      UK  English  Android             1   
4           127  Education      UK  English  Android             0   
5           175  Education      UK  English  Android             0   
6           128       Food      UK    Hindi  Android             0   

   is_verified_creator  
0                    0  
2                    0  
4                    0  
5                    0  
6                    0  
0    0
2    0
4    1
5    0
6    0
Name: watch_completed, dtype: int64


In [19]:
print(y.head())

0    0
2    0
4    1
5    0
6    0
Name: watch_completed, dtype: int64


In [20]:
from sklearn.preprocessing import OneHotEncoder

In [21]:
encoder = OneHotEncoder(
    drop="first",
    sparse_output=False
)

In [25]:
encoded_features = encoder.fit_transform(
    X[["category", "country", "language", "device"]]
)

encoded_df = pd.DataFrame(
    encoded_features,
    columns=encoder.get_feature_names_out(
        ["category", "country", "language", "device"]
    ),
    index=X.index
)

In [26]:
X = X.drop(
    columns=["category", "country", "language", "device"]
)
X = pd.concat(
    [X, encoded_df],
    axis=1
)

In [27]:
from sklearn.model_selection import train_test_split

In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [29]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(399365, 25)
(99842, 25)
(399365,)
(99842,)


In [30]:
from sklearn.linear_model import LogisticRegression

In [33]:
model = LogisticRegression(max_iter=1000)

In [34]:
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [35]:
y_pred = model.predict(X_test)

In [36]:
print(y_pred[:10])

[0 0 0 0 0 0 0 0 0 0]


In [37]:
print(y_test[:10])

488317    0
12739     0
227903    0
535775    0
238384    0
536975    0
451736    0
407955    0
321701    0
216388    0
Name: watch_completed, dtype: int64


In [38]:
from sklearn.metrics import accuracy_score

In [39]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.891528615212035


In [40]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[89012     0]
 [10830     0]]


In [41]:
y.value_counts(normalize=True)

watch_completed
0    0.890987
1    0.109013
Name: proportion, dtype: float64

In [42]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.89      1.00      0.94     89012
           1       0.00      0.00      0.00     10830

    accuracy                           0.89     99842
   macro avg       0.45      0.50      0.47     99842
weighted avg       0.79      0.89      0.84     99842



C:\Users\basav\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\basav\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\basav\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [48]:
model=LogisticRegression(class_weight="balanced", max_iter=1000)

In [49]:
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [50]:
y_pred = model.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[45985 43027]
 [ 5357  5473]]
              precision    recall  f1-score   support

           0       0.90      0.52      0.66     89012
           1       0.11      0.51      0.18     10830

    accuracy                           0.52     99842
   macro avg       0.50      0.51      0.42     99842
weighted avg       0.81      0.52      0.60     99842



In [51]:
merged_df.to_csv("merged_data_updated.csv", index=False)

In [52]:
import os

os.listdir()

['.ipynb_checkpoints',
 '01_data_generation.ipynb',
 '02_data_cleaning.ipynb',
 '03_eda.ipynb',
 '04_Statistics.ipynb',
 '05_Machine_Learning.ipynb',
 'data',
 'images',
 'merged_data.csv',
 'merged_data_updated.csv',
 'models',
 'notebook',
 'README.md',
 'requirements.txt',
 'sql',
 'Tableau']